Classify the documents as one of the following categories
1. Not a (movie/TV show) review
2. Positive (movie/TV show) review
3. Negative (movie/TV show) review

In [1]:
import pandas as pd
import re

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score

from scipy.sparse import hstack

In [2]:
train = pd.read_csv('data/train.csv')

train = train.dropna()

train

,ID,TEXT,LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0
1,11848336500074516230,Just getting released from a six month drug re...,1
2,8453485550425672763,I was greatly dissappointed when I popped this...,0
3,13088897910749354342,This is a film that has garnered any interest ...,2
4,4199320520317837800,This is one of the more adorable episodes of t...,1
...,...,...,...
70312,4095283484534003747,This review owes its existence entirely to a r...,1
70313,15401682080725988675,Even if you could get past the idea that these...,2
70314,9279728677504091897,I grew up watching just about all of the Disne...,1
70315,16755456328208390817,It's difficult to precisely put into words the...,2


In [3]:
def process_ratings(document):
    rating_pattern = r'(\d+)/(10+)'

    def ratingToToken(match):
        num = int(match[1])
        den = int(match[2])

        if num == 0 or den == 0:
            return '<RATING-0>'

        return f'<RATING-{round(num / den) * 10}'
    
    return re.sub(
        rating_pattern,
        ratingToToken,
        document
    )

def prepareData(data):
    X = data['TEXT']
    Y = data['LABEL'] if 'LABEL' in data.columns else None

    # Remove non ascii characters
    X = X.str.replace(r'[^\x00-\x7F]+', '', regex = True)

    X = X.apply(process_ratings)

    # Remove whitespace from ends
    # Add document start and end markers
    X = '<SOS> ' + X.str.strip() + ' <EOS>'

    return X, Y

In [4]:
def prepareCountVectorizer(X):
    sentence_tags = '<SOS>|<EOS>'
    rating_tags = '|'.join(f'<RATING-{i}>' for i in range(11))
    token_pattern = rf'(?u)\b\w\w+\b|(?:{rating_tags})|(?:{sentence_tags})' 
    
    count_vect = CountVectorizer(
        analyzer = 'word',
        token_pattern = token_pattern,
        ngram_range = (1, 2),
        lowercase = True,
        binary = True,
    )
    
    x_counts = count_vect.fit_transform(X)

    return x_counts, count_vect

In [5]:
def stringToDict(document):
    # Remove sos/eos tokens
    doc = document[6:-6]
    return {
        'all_caps': doc.isupper(),
        'has_alpha': any(c.isalpha() for c in doc),
        'num_new_lines': doc.count('\n')
    }

def prepareDictVectorizer(X):
    dict_vect = DictVectorizer()
    
    dict_list = X.map(stringToDict)
    
    x_dict = dict_vect.fit_transform(dict_list)

    return x_dict, dict_vect

In [6]:
def crossValidateModel(X, Y):
    model = LogisticRegression(
        class_weight = 'balanced',
        max_iter = 10000,
    )
    
    scores = cross_val_score(
        model,
        x_features,
        Y,
        cv = 5,
        scoring = 'f1_macro',
        n_jobs = 15,
    )

    print('Scores:', scores)
    print('Mean score:', scores.mean())

    return model

In [7]:
X, Y = prepareData(train)

x_counts, count_vect = prepareCountVectorizer(X)

x_dict, dict_vect = prepareDictVectorizer(X)

x_features = hstack([x_counts, x_dict])

model = crossValidateModel(x_features, Y)

# Current best 0.9213747247075001

Scores: [0.91756449 0.92462947 0.92171036 0.91954213 0.92342717]
Mean score: 0.9213747247075001


# Prediction

In [ ]:
# Train model on all data
model.fit(x_features, Y)

In [ ]:
test = pd.read_csv('data/test.csv')

x_test, _ = prepareData(test)

x_test_counts = count_vect.transform(x_test)
x_test_dict = dict_vect.transform(
        x_test.map(stringToDict)
    )

x_test_features = hstack([x_test_counts, x_test_dict])

y_pred = model.predict(x_test_features)

In [ ]:
test['LABEL'] = y_pred

In [ ]:
test[['ID', 'LABEL']].to_csv('submission.csv', index = False)